# Projet 2 · Nettoyer, construire un dashboard et entraîner un premier modèle · ⭐⭐

**Bloc 2 · Niveau ⭐⭐ Intermédiaire** · Le projet du « métier de data scientist ».

Tu reprends le dataset Kaggle du projet 1 et tu fais le travail complet d'un data scientist : **nettoyer** (et documenter chaque correction), construire un **dashboard Plotly** de 3 graphiques, entraîner un **premier modèle scikit-learn** qui prédit quelque chose d'utile, puis **présenter** le tout à un « client » en 2 minutes.

Comment travailler :
- Google Colab, `Maj + Entrée`. Les cellules « À toi » s'exécutent déjà avec un exemple à remplacer.
- Chaque étape correspond à une séance du bloc 2 : nettoyage (S4), dashboard (S5), modèle (S6).


## Préparation : le même dataset qu'au projet 1

Même variable `DATASET`, même miroir public. La question que le modèle devra répondre dépend du dataset :

| `DATASET` | Ce que le modèle prédit (`cible`) | À partir de (`features`) |
|---|---|---|
| jeux_video | le jeu a-t-il **plus de 100 000 propriétaires** ? | prix, année, temps de jeu |
| netflix | est-ce un **film** (1) ou une **série** (0) ? | année, nombre de genres, longueur de la description, taille du casting |
| spotify | le morceau est-il **populaire** (popularité ≥ 50) ? | danceability, energy, valence, tempo, durée, année... |
| pokemon | le Pokémon est-il **légendaire** ? | PV, attaque, défense, vitesse, génération |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATASET = "jeux_video"   # ← choisis : "jeux_video", "netflix", "spotify" ou "pokemon"
FICHIER_LOCAL = None     # ← ex. "vgsales.csv" si tu as déposé le fichier Kaggle dans Colab (sinon laisse None)

DATASETS = {
    "jeux_video": {
        "nom": "Jeux vidéo sur Steam (prix, joueurs, temps de jeu, notes)",
        "kaggle": "https://www.kaggle.com/datasets/gregorut/videogamesales",
        "miroir": "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2019/2019-07-30/video_games.csv",
    },
    "netflix": {
        "nom": "Catalogue Netflix (films et séries)",
        "kaggle": "https://www.kaggle.com/datasets/shivamb/netflix-shows",
        "miroir": "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-04-20/netflix_titles.csv",
    },
    "spotify": {
        "nom": "30 000 morceaux Spotify (popularité, genre, caractéristiques audio)",
        "kaggle": "https://www.kaggle.com/datasets/joebeachcapital/30000-spotify-songs",
        "miroir": "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv",
    },
    "pokemon": {
        "nom": "800 Pokémon (types, statistiques, génération)",
        "kaggle": "https://www.kaggle.com/datasets/abcsds/pokemon",
        "miroir": "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv",
    },
}
info = DATASETS[DATASET]

try:
    if FICHIER_LOCAL:
        df = pd.read_csv(FICHIER_LOCAL)
        print("Fichier local chargé :", FICHIER_LOCAL)
    else:
        df = pd.read_csv(info["miroir"])
        print("Miroir public chargé (mêmes données que la page Kaggle)")
except Exception as erreur:
    print("Impossible de charger les données : pas de réseau ?", erreur)
    raise

print("Dataset :", info["nom"])
print("Page Kaggle :", info["kaggle"])
print(df.shape[0], "lignes ×", df.shape[1], "colonnes")
df.head()

## 1. Nettoyer et documenter (séance 4, ~40 min)

Un vrai dataset est sale : cases vides, doublons, texte là où on attend un nombre. La règle du métier : **chaque correction est notée** dans un journal, pour que quelqu'un d'autre (ou toi dans 3 mois) comprenne ce qui a été fait.

La fonction `nettoyer` ci-dessous fait les corrections de base pour le dataset choisi et remplit `JOURNAL`. Lis-la, puis ajoute au moins **une correction à toi** dans la cellule « À toi ».

In [ ]:
JOURNAL = []   # une ligne par correction : (colonne, problème, ce qu'on a fait, lignes concernées)

def noter(colonne, probleme, action, n):
    JOURNAL.append({"colonne": colonne, "problème": probleme, "action": action, "lignes": int(n)})

def nettoyer(df, dataset):
    df = df.copy()
    n_avant = len(df)
    df = df.drop_duplicates()
    noter("(toutes)", "doublons", "supprimés", n_avant - len(df))

    if dataset == "jeux_video":
        df["annee"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year
        noter("release_date", "texte « Nov 16, 2004 »", "converti en année (annee)", df["annee"].isna().sum())
        df["proprietaires_min"] = df["owners"].str.extract(r"^([\d,]+)")[0].str.replace(",", "").astype(float)
        noter("owners", "fourchette texte « 20,000 .. 50,000 »", "bas de la fourchette en nombre (proprietaires_min)", len(df))
        n = df["price"].isna().sum()
        df["price"] = df["price"].fillna(0)
        noter("price", "prix manquant", "remplacé par 0 (jeux gratuits)", n)
        df = df.dropna(subset=["annee", "average_playtime", "median_playtime"])
        noter("annee, playtime", "valeurs manquantes", "lignes supprimées", n_avant - len(df))
        df["cible"] = (df["proprietaires_min"] >= 100_000).astype(int)
        df["categorie"], df["valeur"], df["temps"] = df["publisher"], df["price"], df["annee"]
        FEATURES = ["price", "annee", "average_playtime", "median_playtime"]

    elif dataset == "netflix":
        df["annee_ajout"] = pd.to_datetime(df["date_added"], errors="coerce").dt.year
        n = df["annee_ajout"].isna().sum()
        df["annee_ajout"] = df["annee_ajout"].fillna(df["release_year"])
        noter("date_added", "date manquante", "remplacée par l'année de sortie", n)
        for col in ["director", "cast", "country"]:
            noter(col, "valeur manquante", "remplacée par « Inconnu »", df[col].isna().sum())
            df[col] = df[col].fillna("Inconnu")
        df["genre_principal"] = df["listed_in"].str.split(",").str[0].str.strip()
        df["nb_genres"] = df["listed_in"].str.count(",") + 1
        df["nb_acteurs"] = df["cast"].str.count(",") + 1
        df["longueur_description"] = df["description"].str.len()
        df["cible"] = (df["type"] == "Movie").astype(int)
        df["categorie"], df["valeur"], df["temps"] = df["genre_principal"], df["nb_genres"], df["release_year"]
        FEATURES = ["release_year", "annee_ajout", "nb_genres", "nb_acteurs", "longueur_description"]

    elif dataset == "spotify":
        n = df["track_name"].isna().sum()
        df = df.dropna(subset=["track_name", "track_artist"])
        noter("track_name", "titre manquant", "lignes supprimées", n)
        n = df.duplicated(subset=["track_id"]).sum()
        df = df.drop_duplicates(subset=["track_id"])
        noter("track_id", "même morceau dans plusieurs playlists", "un seul exemplaire gardé", n)
        df["annee"] = df["track_album_release_date"].str[:4].astype(int)
        df["duree_min"] = df["duration_ms"] / 60000
        df["cible"] = (df["track_popularity"] >= 50).astype(int)
        df["categorie"], df["valeur"], df["temps"] = df["playlist_genre"], df["track_popularity"], df["annee"]
        FEATURES = ["danceability", "energy", "loudness", "speechiness", "acousticness",
                    "instrumentalness", "liveness", "valence", "tempo", "duree_min", "annee"]

    elif dataset == "pokemon":
        n = df["Type 2"].isna().sum()
        df["Type 2"] = df["Type 2"].fillna("Aucun")
        noter("Type 2", "pas de second type", "remplacé par « Aucun »", n)
        df["cible"] = df["Legendary"].astype(int)
        df["categorie"], df["valeur"], df["temps"] = df["Type 1"], df["Total"], df["Generation"]
        FEATURES = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed", "Generation"]

    return df, FEATURES

df, FEATURES = nettoyer(df, DATASET)
print(len(df), "lignes après nettoyage · cible positive dans", round(df["cible"].mean() * 100, 1), "% des cas")
pd.DataFrame(JOURNAL)

**À toi** : ajoute une correction. Idées : supprimer les lignes où `valeur` est aberrante (prix > 200 €, durée > 20 min, Total > 700...), harmoniser les majuscules d'une colonne texte (`.str.title()`), retirer les espaces (`.str.strip()`). N'oublie pas `noter(...)`.

<details><summary>Solution (exemple sur les valeurs aberrantes)</summary>

```python
seuil = df["valeur"].quantile(0.99)
n = (df["valeur"] > seuil).sum()
df = df[df["valeur"] <= seuil]
noter("valeur", f"valeurs extrêmes (> {seuil:.1f})", "lignes supprimées", n)
pd.DataFrame(JOURNAL)
```
</details>

In [ ]:
# À toi : au moins une correction supplémentaire
seuil = df["valeur"].quantile(0.99)
n = (df["valeur"] > seuil).sum()
df = df[df["valeur"] <= seuil]
noter("valeur", f"valeurs extrêmes (> {seuil:.1f})", "lignes supprimées", n)
pd.DataFrame(JOURNAL)

## 2. Le dashboard Plotly (séance 5, ~40 min)

Trois graphiques **interactifs** (survole, zoome, clique sur la légende) qui racontent une histoire : un **constat**, une **preuve**, une **recommandation**. Chaque graphique doit répondre à une question écrite en titre.

In [ ]:
import plotly.express as px

# Graphique 1 : les 10 catégories les plus fréquentes
top = df["categorie"].value_counts().head(10).reset_index()
top.columns = ["categorie", "nombre"]
fig1 = px.bar(top, x="nombre", y="categorie", orientation="h",
              title=f"Top 10 des catégories ({DATASET})")   # ← mets ta question en titre
fig1.show()

In [ ]:
# Graphique 2 : la cible selon la catégorie (part de « oui » dans chaque catégorie fréquente)
frequentes = df["categorie"].value_counts()
frequentes = frequentes[frequentes >= 30].index
part = (df[df["categorie"].isin(frequentes)].groupby("categorie")["cible"].mean() * 100).sort_values().reset_index()
part.columns = ["categorie", "part_cible_%"]
fig2 = px.bar(part.tail(12), x="part_cible_%", y="categorie", orientation="h", color="part_cible_%",
              title="Dans quelles catégories la cible est-elle la plus fréquente ? (%)")
fig2.show()

In [ ]:
# Graphique 3 : l'évolution dans le temps, avec le détail au survol
evolution = df.groupby("temps").agg(nombre=("cible", "size"), part_cible=("cible", "mean")).reset_index()
evolution = evolution[evolution["nombre"] >= 10]
fig3 = px.line(evolution, x="temps", y="nombre", markers=True, hover_data=["part_cible"],
               title="Combien de lignes par période ? (survole pour la part de cible)")
fig3.show()

**À toi** : remplace **au moins un** des trois graphiques par un graphique de ton choix qui sert ton histoire. Idées Plotly : `px.scatter(df, x=..., y=..., color="cible")`, `px.box(df, x="categorie", y="valeur")`, `px.histogram(df, x="valeur", color="cible")`. Puis écris ton pitch.

In [ ]:
# À toi : ton graphique
fig4 = px.histogram(df, x="valeur", color="cible", barmode="overlay", nbins=40,
                    title="La cible se voit-elle sur la distribution de « valeur » ?")
fig4.show()

In [ ]:
PITCH = {
    "constat":        "Ex. : 3 éditeurs sortent la moitié des jeux",              # ← ce que montrent les données
    "preuve":         "Ex. : graphique 1 (barres) et graphique 2 (part de la cible)",  # ← quel graphique le prouve
    "recommandation": "Ex. : si on veut être joué, viser un temps de jeu médian > 2h",  # ← ce que le client devrait faire
}
for k, v in PITCH.items():
    print(f"{k:15s}: {v}")

## 3. Premier modèle (séance 6, ~50 min)

On montre au modèle des exemples **avec** la réponse (`cible`), il trouve la règle, puis on le teste sur des lignes qu'il n'a **jamais vues**. Trois étapes : séparer train/test, comparer avec un modèle « bête » (il prédit toujours la classe majoritaire), puis deux vrais modèles.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

X = df[FEATURES].fillna(df[FEATURES].median())
y = df["cible"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Entraînement :", len(X_train), "lignes · Test :", len(X_test), "lignes")
print("Variables utilisées :", FEATURES)

modeles = {
    "bête (classe majoritaire)": DummyClassifier(strategy="most_frequent"),
    "régression logistique":     make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "forêt aléatoire":           RandomForestClassifier(n_estimators=200, random_state=42),
}
resultats = {}
for nom, modele in modeles.items():
    modele.fit(X_train, y_train)
    resultats[nom] = accuracy_score(y_test, modele.predict(X_test))
    print(f"{nom:28s} exactitude sur le test : {resultats[nom]*100:.1f} %")

L'**exactitude** seule peut tromper : si 92 % des Pokémon ne sont pas légendaires, le modèle bête a déjà 92 %. On regarde donc la **matrice de confusion** : combien de vrais « oui » sont trouvés.

In [ ]:
meilleur_nom = max(resultats, key=resultats.get)
meilleur = modeles[meilleur_nom]
pred = meilleur.predict(X_test)
mc = confusion_matrix(y_test, pred)
print("Meilleur modèle :", meilleur_nom)
print(pd.DataFrame(mc, index=["vrai : non", "vrai : oui"], columns=["prédit : non", "prédit : oui"]))
rappel = mc[1, 1] / mc[1].sum() if mc[1].sum() else 0
print(f"Parmi les vrais « oui », le modèle en trouve {rappel*100:.0f} %")

In [ ]:
# Quelles variables comptent le plus ? (forêt aléatoire)
foret = modeles["forêt aléatoire"]
importances = pd.Series(foret.feature_importances_, index=FEATURES).sort_values()
fig5 = px.bar(importances, orientation="h", title="Importance des variables pour la forêt")
fig5.show()

**À toi** : améliore le modèle. Deux pistes : (1) ajouter une variable à `FEATURES` (une colonne numérique du dataset, ou une que tu crées, ex. `df["prix_par_heure"] = df["price"] / (df["average_playtime"] + 1)`) ; (2) changer `n_estimators` ou `max_depth` de la forêt. Note l'exactitude avant/après dans `ESSAIS`.

<details><summary>Indice</summary>

```python
df["nouvelle"] = ...                     # ta variable
FEATURES2 = FEATURES + ["nouvelle"]
X2 = df[FEATURES2].fillna(df[FEATURES2].median())
X_tr, X_te, y_tr, y_te = train_test_split(X2, y, test_size=0.2, random_state=42, stratify=y)
f2 = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42).fit(X_tr, y_tr)
print(accuracy_score(y_te, f2.predict(X_te)))
```
</details>

In [ ]:
# À toi
ESSAIS = [
    {"essai": "forêt de base", "exactitude": round(resultats["forêt aléatoire"], 3)},
]
pd.DataFrame(ESSAIS)

In [ ]:
CE_QUE_LE_MODELE_A_COMPRIS = "Ex. : plus un jeu est joué longtemps, plus il a de chances d'avoir beaucoup de propriétaires."   # ← ta phrase
print(CE_QUE_LE_MODELE_A_COMPRIS)

## 4. La présentation « client » (2 minutes chrono)

Structure imposée, 4 phrases : **le problème** (quelle question du client ?), **les données** (d'où, combien, ce qu'on a nettoyé), **le résultat** (un graphique + l'exactitude du modèle, comparée au modèle bête), **la recommandation**. Le reste du groupe joue le client et pose une question piège (« et si les données sont biaisées ? »).

In [ ]:
def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans arrêter le notebook."""
    print(("✅ " if condition else "❌ ") + nom)

verifier("Au moins 2 corrections dans le journal de nettoyage", len(JOURNAL) >= 2)
verifier("Le pitch est rempli (pas les exemples)", all(not v.startswith("Ex.") for v in PITCH.values()))
verifier("Le meilleur modèle bat le modèle bête", resultats[meilleur_nom] > resultats["bête (classe majoritaire)"])
verifier("Au moins 2 essais notés", len(ESSAIS) >= 2)
verifier("Une phrase explique ce que le modèle a compris", not CE_QUE_LE_MODELE_A_COMPRIS.startswith("Ex."))

## Pour aller plus loin
- Transforme le dashboard en application **Streamlit** (`st.plotly_chart(fig1)`) et déploie-la gratuitement sur Streamlit Community Cloud depuis ton GitHub.
- Essaie une **régression** : prédire la valeur elle-même (prix, popularité, Total) avec `RandomForestRegressor` et mesurer l'erreur moyenne.
- Compare avec un notebook Kaggle sur ton dataset (liens dans le README) : quelles variables ont-ils créées que tu n'as pas ?